In [22]:
from astropy.io import fits
from scipy.io import readsav
import numpy as np
import os

from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim

import tools as t

In [2]:
# load files
path_data = '../data/sim_series/'  # directory where the files are saved
N_files = 20  # 20 temporal series (from 0 to 19)
data_spectra = []
data_rv = []
for i in range(N_files):  # files have the same name, only number changes
    data_spectra.append(fits.getdata(path_data + f'flux_norm_6173_lionel_tau1_spatialscaling1_sansrotation_serie{i}.fits'))
    data_rv.append(readsav(path_data + f'res_rv_bis_6173_lionel_tau1_scaling1_sansrot_serie{i}.sav'))
data_spectra = np.array(data_spectra)
data_rv = np.array(data_rv)
data_rv[0].keys()

dict_keys(['rv_gauss', 'rv_correl', 'rv_deriv', 'xbiss', 'ybiss', 'l_true', 'lambd', 'flux_moy'])

In [3]:
# extract data
wl = np.array(data_rv[0]['lambd'])  # same wavelength grid for all time series (verified)
times = np.arange(0, len(data_spectra[0]) * 144, 144) / 60 / 24  # time in days, same for all time series
times_sec = times * 86400  # time in sec
# extract RV data
rv_ccf = []
for i in range(N_files):
    rv_ccf.append(data_rv[i]['rv_correl'])
wl = np.array(wl)
rv_ccf = np.array(rv_ccf)

### Preprocessing

In [4]:
# calculate the mean spectra
mean_spec = np.array([np.mean(i, axis=0) for i in data_spectra])  # list of mean spectrum for each TS
d_S0 = np.array([np.gradient(i, wl) for i in mean_spec])  # list of derivative of the mean spectra for each TS

In [5]:
c = 3e8  # speed of light [m/s]
S_f = []  # array of matrices
D_vectors = []  # array of Doppler vectors

for i in range(N_files):  # for each TS
    s, v = t.subtract_D(wl, data_spectra[i], c=c)  # take out projection over the Doppler vector
    S_f.append(s)
    D_vectors.append(v)

S_f = np.array(S_f)
D_vectors = np.array(D_vectors)

In [6]:
# calculate radial velocity by template matching
C_0 = 1000  # continuum constant
rv_tm = []  # list of RV vector for each TS
for j in range(N_files):  # for each TS
    rv_j = []
    for k in range(len(times)):  # for each spectrum
        rv_j.append(t.calculate_rv(wl, mean_spec[j], data_spectra[j][k], C_0))  # calculate RV by TM
    rv_tm.append(np.array(rv_j))
rv_tm = np.array(rv_tm)

## Apply PCA

In [7]:
# apply double centering to each S_f matrix
spectra_centered = np.array([t.double_centering(i) for i in S_f])

In [8]:
# apply PCA to the centered matrix
n_components = 20

pcs_ts = []  # list of N_files lists of of principal components
loadings_ts = []  # list of loadings of all PCs for each TS
scores_ts = []  # list of scores of all PCs for each TS

for i in spectra_centered:  # each i is a double-centered matrix (corresponding to each TS)
    pca_flux_i = PCA(n_components=n_components)  # create PCA object
    pcs_i = pca_flux_i.fit_transform(i)  # fit the model with spectra_centered and apply the dimensionality reduction
    pcs_ts.append(pcs_i)  # save PCs
    loadings_ts.append(pca_flux_i.components_)  # save loadings
    scores_ts.append(np.array([pcs_i[:, j] for j in range(n_components)]))  # save scores

pcs_ts = np.array(pcs_ts)
loadings_ts = np.array(loadings_ts)
scores_ts = np.array(scores_ts)

In [9]:
# calculate median correlation coefficient of each PC (noisy case)

# create noisy TS
sigma = 1e-2

ts_noisy = data_spectra + np.random.normal(
    loc=0,
    scale=sigma,
    size=data_spectra.shape
)

# calculate rv
mean_spec_noisy = np.array([np.mean(i, axis=0) for i in ts_noisy])  # list of mean spectrum for each TS
C_0 = 1000  # continuum constant
rv_noisy = []  # list of RV vector for each TS
for j in range(N_files):  # for each TS
    rv_j = []
    for k in range(len(times)):  # for each spectrum
        rv_j.append(t.calculate_rv(wl, mean_spec_noisy[j], ts_noisy[j][k], C_0))  # calculate RV by TM
    rv_noisy.append(np.array(rv_j))
rv_noisy = np.array(rv_noisy)

# preprocessing
c = 3e8
S_noisy = []
D_noisy = []
for i in range(N_files):  # for each TS
    s, v = t.subtract_D(wl, ts_noisy[i], c=c)  # take out projection over the Doppler vector
    S_noisy.append(s)
    D_noisy.append(v)
spectra_noisy_centered = np.array([t.double_centering(i) for i in S_noisy])  # apply double centering to each S_noisy matrix

# PCA
pcs_noisy = []  # list of N_files lists of of principal components
loadings_noisy = []  # list of loadings of all PCs for each TS
scores_noisy = []  # list of scores of all PCs for each TS

for i in spectra_noisy_centered:  # each i is a double-centered matrix (corresponding to each TS)
    pca_flux_i = PCA(n_components=n_components)  # create PCA object
    pcs_i = pca_flux_i.fit_transform(i)  # fit the model with spectra_centered and apply the dimensionality reduction
    pcs_noisy.append(pcs_i)  # save PCs
    loadings_noisy.append(pca_flux_i.components_)  # save loadings
    scores_noisy.append(np.array([pcs_i[:, j] for j in range(n_components)]))  # save scores

pcs_noisy = np.array(pcs_noisy)
loadings_noisy = np.array(loadings_noisy)
scores_noisy = np.array(scores_noisy)

In [10]:
scores_ts[0].shape

(20, 3686)

In [11]:
ex = 0  # first TS as example
loadings_ex = loadings_ts[ex]  # loadings of the PCs of the selected TS
spectra_ex = ts_noisy[ex]  # example TS of noisy spectra
indicators = {}  # each of the n_components elements in this dict is a list of size 3686, and each element in that list is a list of size 12
indicators[0] = rv_tm[0]  # add RVs as I_0

for i in range(n_components):  # iterate over the number of components
    L_i = loadings_ex[i]
    indicators[i+1] = []
    for S_j in spectra_ex:
        I_ij = np.dot(L_i.T, S_j) / np.linalg.norm(L_i)**2
        indicators[i+1].append(I_ij)
print(f'{len(indicators)} indicators of size {len(indicators[1])}.')
print(indicators.keys())

21 indicators of size 3686.
dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20])


In [14]:
# define data for neural network
X = torch.tensor(scores_ts[ex].T).float()  # transposing and converting to float32
y = torch.tensor(np.array([rv_tm[ex]]).T).float()
print(f'Shape of X: {X.shape}')
print(f'Shape of y: {y.shape}')

Shape of X: torch.Size([3686, 20])
Shape of y: torch.Size([3686, 1])


In [ ]:
N = 3686
M = 20

# -------------------------
# Model
# -------------------------

model = nn.Sequential(
    nn.Linear(M, 20),
    nn.Linear(20, 10),
    nn.Linear(10, 1)
)


# -------------------------
# Loss and optimizer
# -------------------------

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)


# -------------------------
# Training
# -------------------------
iterations = 1000

for epoch in range(iterations):

    # Forward pass
    y_pred = model(X)

    # Calculate loss
    loss = criterion(y_pred, y)

    # Clear old gradients
    optimizer.zero_grad()

    # Calculate gradients
    loss.backward()

    # Update weights
    optimizer.step()

    if epoch % 100 == 0 or epoch == iterations - 1:
        print(f"Epoch {epoch}: loss = {loss.item():.4f}")


print("Input shape: ", X.shape)
print("Output shape:", model(X).shape)

Epoch 0: loss = 0.8995
Epoch 100: loss = 0.8958
Epoch 200: loss = 0.8958
Epoch 300: loss = 0.8958
Epoch 400: loss = 0.8958
Epoch 500: loss = 0.8958
Epoch 600: loss = 0.8958
Epoch 700: loss = 0.8958
Epoch 800: loss = 0.8958
Epoch 900: loss = 0.8958
Epoch 999: loss = 0.8958
Input shape:  torch.Size([3686, 20])
Output shape: torch.Size([3686, 1])
